# 12.01 · When to fine-tune (and when not)

**Chapter:** Fine-tuning & Post-training · **Milestone:** M6

## Goals
- [ ] Know what fine-tuning changes (behaviour, format, style, narrow skills) and what it doesn't (fresh knowledge → use RAG)
- [ ] Walk the decision ladder: prompt → few-shot → RAG → bigger model → fine-tune → distil
- [ ] Measure baselines on the chapter's task before training anything

## Resources
- [Maxime Labonne – LLM Course (fine-tuning section)](https://github.com/mlabonne/llm-course) — free
- [OpenAI – Model optimization guide](https://platform.openai.com/docs/guides/model-optimization) — free
- [Anyscale – Fine-tuning is for form, not facts](https://www.anyscale.com/blog/fine-tuning-is-for-form-not-facts) — free

**The chapter's task** (`task.py` next to this notebook): extract `{"date", "amount", "currency"}` as exact JSON from messy English/Spanish
payment messages ("me cobraron S/765.95 el 6 de abril de 2026"). Labels are generated programmatically → exact, free, unlimited.

In [ ]:
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from aieng.llm import client
from task import SYSTEM, dataset, messages, score

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
test = dataset(60, seed=2)
for e in test[:4]:
    print(e["text"], "->", e["target"])

## 1. The decision ladder
| Step | Fixes | Cost |
|---|---|---|
| 1. Better prompt / structured output | instructions, format | minutes |
| 2. Few-shot examples | format, edge cases | more input tokens per call |
| 3. RAG | missing / fresh / private **knowledge** | retrieval infra |
| 4. Bigger or better model | capability | higher price & latency per call |
| 5. **Fine-tune** (LoRA/SFT) | consistent format/style/behaviour, narrow skills, *smaller model at the quality of a bigger one* | data + training + hosting |
| 6. Preference / RL tuning | subtle preferences, verifiable skills | more data, more care |
| 7. Distillation | shrink a big model's behaviour into a small one | teacher calls + training |

Fine-tune when: the task is **narrow and stable**, you have (or can make) **hundreds+ good examples**, prompting plateaus, or you need a
**small/cheap/local/private** model to match a big one. Don't fine-tune to add facts that change — that's RAG's job.

## 2. Baselines
### a. A tiny local model, zero-shot (the one we'll fine-tune in 12.03)

In [ ]:
SMALL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(SMALL)
small = AutoModelForCausalLM.from_pretrained(SMALL).to(device).eval()


def hf_generate(model, tok, msgs, max_new_tokens=48):
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)


def evaluate(generate_fn, rows):
    start, scores = time.perf_counter(), []
    for e in rows:
        try:
            scores.append(score(generate_fn(messages(e["text"])), e["target"]))
        except Exception as err:  # API errors / rate limits count as failures
            scores.append(score("", e["target"]))
    agg = {k: round(sum(s[k] for s in scores) / len(scores), 2) for k in scores[0]}
    return agg, (time.perf_counter() - start) / len(rows)


print("SmolLM2-135M zero-shot:", evaluate(lambda m: hf_generate(small, tok, m), test))
print("example output:", hf_generate(small, tok, messages(test[0]["text"])))

### b. A 12× bigger local model (qwen3:1.7b via Ollama), zero-shot and few-shot

In [ ]:
oc, OMODEL = client("ollama")


def ollama_fn(msgs):
    return oc.chat.completions.create(model=OMODEL, messages=msgs, temperature=0, max_tokens=80, reasoning_effort="none").choices[0].message.content


FEW_SHOT = [m for e in dataset(4, seed=99) for m in messages(e["text"], e["target"])[1:]]  # user/assistant pairs, not from the test set


def few_shot(fn):
    return lambda msgs: fn([msgs[0], *FEW_SHOT, msgs[1]])


print("qwen3:1.7b zero-shot:", evaluate(ollama_fn, test))
print("qwen3:1.7b 4-shot   :", evaluate(few_shot(ollama_fn), test))

### c. A hosted model (optional; runs if a Gemini/Groq key is set — 20 examples to respect free-tier limits)

In [ ]:
for provider in ["gemini", "groq"]:
    key = {"gemini": "GEMINI_API_KEY", "groq": "GROQ_API_KEY"}[provider]
    if not os.getenv(key):
        continue
    hc, hmodel = client(provider)
    fn = lambda msgs, hc=hc, hmodel=hmodel: hc.chat.completions.create(model=hmodel, messages=msgs, temperature=0, max_tokens=200).choices[0].message.content
    print(f"{provider} {hmodel}:", evaluate(fn, test[:20]))

## 3. Read the baselines
Typical result: the 135M model is near zero; the 1.7B model does much better and few-shot helps; hosted frontier models are near-perfect.
**The fine-tuning question**: can the 135M model — ~100× cheaper to serve than a hosted API, runs on a phone, private — reach the hosted
model's accuracy? 12.03 answers it. If a cheap hosted model at your volume costs little, the honest answer may be "just use the API".

## 4. Cost sketch

In [ ]:
calls_per_day = 50_000
in_tok, out_tok = 120, 25
for name, p_in, p_out in [("hosted small model (illustrative $0.10/$0.40 per 1M)", 0.10, 0.40),
                          ("hosted mid model (illustrative $1/$4 per 1M)", 1.0, 4.0)]:
    print(f"{name:52} ${calls_per_day * 30 * (in_tok * p_in + out_tok * p_out) / 1e6:8,.0f}/month")
print(f"{'self-hosted 135M on a small CPU VM (illustrative)':52} ${'~20-40':>8}/month + your time")

## Exercises
1. Write a one-page decision memo for three products: (a) legal-contract Q&A over a firm's documents, (b) a support bot that must answer in
   the company's tone, (c) on-device receipt parsing for a mobile app. Prompt, RAG, fine-tune, or a mix?
2. Improve the qwen3:1.7b baseline with prompting alone (better instructions, structured outputs from 05.03). How close to 100% can you get?
3. Find two failure categories in the 135M model's outputs (read 10 of them). Predict which ones fine-tuning will fix.

In [ ]:
# your code here